# 06 — CYP2D6 Outlier Check

**Roadmap Step 1**: does excluding CYP2D6 compounds
flagged by two independent, model-choice-agnostic criteria change CV performance for
the four CYP2D6-relevant configs from `05_cv_comparison`? All flagging and retraining
logic lives in `scripts/cyp2d6_outlier_check.py` (CLAUDE.md: scripts train, notebooks
report/analyze) -- **this notebook only loads and reports its already-completed
outputs; nothing here retrains or re-runs anything.**

**Two independent flagging criteria** (both computed from existing 05 artifacts --
no new training for the flagging step itself):

1. **CV-residual**: pooled out-of-fold |y_pred - y_true| for CYP2D6, from
   `chemprop_chemeleoninit`'s 25 (5 repeat x 5 fold) OOF prediction files only --
   deliberately not all 12 configs, so outlier status is model-independent once
   frozen. Threshold **confirmed by the user** after reviewing a candidate-threshold
   report: flag `mean_abs_residual` > the 95th percentile of the pooled distribution.
2. **CI-width**: flag the top 5% of CYP2D6 compounds by
   `(conf_high - conf_low)`, already in `train_inhibition_curated.csv`. As `04c`
   already found CI-width correlates with residual on every isoform, this criterion
   can only show whether removing noisy CYP2D6 points helps CYP2D6 -- not why CYP2D6
   specifically underperforms.

Threshold confirmation happened **before** any retraining (non-negotiable per the
task spec, to keep this a real test rather than data-dredging) -- flagged-compound
counts and overlap were reported and confirmed first; only then did retraining run.

**Exclusion design** (also confirmed with the user before retraining): exclusion is
**train-only** -- a flagged compound is dropped from whichever fold's training pool
it would have been part of, but still scored normally whenever it lands in a fold's
held-out test set. This keeps the "before" and "after" evaluation population
identical (the same 1,493 CYP2D6-labeled compounds, the same 25 folds), so any metric
change reflects the retrained model, not a smaller/easier test set. For
`chemprop_chemeleoninit`'s multitask model, exclusion masks only the CYP2D6 label for
flagged compounds -- they keep contributing to CYP1A2/2C9/3A4 training.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from scipy.stats import shapiro, ttest_rel, wilcoxon

OUT = REPO_ROOT / "outputs" / "06_outlier_check"
FLAGGED_DIR = OUT / "flagged_compounds"
SCORE_DIR_06 = OUT / "scores"
SUMMARY_05_PATH = REPO_ROOT / "outputs" / "05_cv_comparison" / "summary_table.csv"

CONFIGS = ["chemeleon__rf", "chemeleon__lightgbm", "ecfp4_narrow__lightgbm", "chemprop_chemeleoninit"]
CRITERIA = ["residual", "ci_width"]
METRICS = ["ST-RAE", "R2"]
ALPHA = 0.05
N_CYP2D6_LABELED = 1493  # confirmed against train_inhibition_curated.csv in the flagging script's own log

print(f"python: {sys.version.split()[0]}")
print(f"configs: {CONFIGS}")
print(f"criteria: {CRITERIA}")

python: 3.11.13
configs: ['chemeleon__rf', 'chemeleon__lightgbm', 'ecfp4_narrow__lightgbm', 'chemprop_chemeleoninit']
criteria: ['residual', 'ci_width']


## 1. Flagging results

Both flagged-compound lists were written by `scripts/cyp2d6_outlier_check.py` to
`outputs/06_outlier_check/flagged_compounds/` before any retraining. Loaded here, not
recomputed.

In [2]:
residual_flagged = pd.read_csv(FLAGGED_DIR / "criterion1_residual_flagged.csv")
ci_flagged = pd.read_csv(FLAGGED_DIR / "criterion2_ci_width_flagged.csv")

overlap = set(residual_flagged["inchikey"]) & set(ci_flagged["inchikey"])
smaller = min(len(residual_flagged), len(ci_flagged))

flag_summary = pd.DataFrame([
    {"criterion": "1. CV-residual (mean_abs_residual > p95, chemprop_chemeleoninit OOF)",
     "n_flagged": len(residual_flagged), "pct_of_1493": len(residual_flagged) / N_CYP2D6_LABELED},
    {"criterion": "2. CI-width (top 5% of conf_high - conf_low)",
     "n_flagged": len(ci_flagged), "pct_of_1493": len(ci_flagged) / N_CYP2D6_LABELED},
])
print(flag_summary.to_string(index=False))
print(f"\noverlap: {len(overlap)} compounds ({len(overlap) / smaller:.2%} of the smaller 75-compound set)")

                                                           criterion  n_flagged  pct_of_1493
1. CV-residual (mean_abs_residual > p95, chemprop_chemeleoninit OOF)         75     0.050234
                        2. CI-width (top 5% of conf_high - conf_low)         75     0.050234

overlap: 26 compounds (34.67% of the smaller 75-compound set)


**What this shows:** criterion 1 (CV-residual, p95) and criterion 2 (CI-width, top
5%) each flag 75 of the 1,493 CYP2D6-labeled compounds (5.02% -- p95 of a
1,493-compound distribution and a fixed 5% quantile land on the same count here by
coincidence of the sample size, not by construction). 26 compounds (34.7% of the
smaller 75-compound set) are flagged by both -- a real but partial overlap, consistent
with 04c's finding that CI-width correlates with residual without being the same
signal: most of what each criterion flags, the other does not.

## 2. Before/after CV comparison

**Statistical method -- a confirmed deviation from 05, not silently adapted.** 05's
Levene's -> RM-ANOVA/Tukey-or-Friedman/Conover-Friedman pipeline (with BH correction
above 10 groups and a 5-metric Bonferroni gate) was built for a many-config, 5-metric
cross-sectional screen, where Levene's homogeneity-of-variance check makes sense
across *independent* groups. This comparison is different in kind: for each
(config, criterion), "baseline" and "excluded" are the **same 25 folds**, retrained
with vs. without the flagged compounds -- a paired, not independent, design. Levene's
test is the wrong assumption check for that (flagged by the user before this section
was written, in place of the author's first proposal to reuse Levene's-based
branching literally). Method actually used, per (config, criterion, metric):

1. Compute the 25 per-fold deltas (excluded - baseline).
2. Shapiro-Wilk test on those 25 deltas.
3. Paired t-test (on the raw before/after arrays) if Shapiro-Wilk does not reject
   normality (p >= 0.05); Wilcoxon signed-rank on the deltas otherwise.

No BH correction and no cross-metric Bonferroni gate (both calibrated for 05's joint
5-metric, many-group screen, not this 2-metric before/after check) -- confirmed with
the user. Only ST-RAE and R2 are tested (the two metrics the task asks about); each
(config, criterion) pair gets its own test at alpha=0.05, uncorrected across the 4
configs x 2 criteria -- with 16 total tests below, ~0.8 false positives are expected
under the null by chance alone, worth keeping in mind when reading the table.

"Before" point estimates -- the mean of each fold's 1,000-sample bootstrap
distribution, matching 05's own aggregation convention exactly -- come from 05's own
`summary_table.csv`, unchanged. "After" point estimates are aggregated the identical
way from `scripts/cyp2d6_outlier_check.py`'s own per-fold score files.

In [3]:
rows = []
for criterion in CRITERIA:
    for config in CONFIGS:
        for repeat in range(5):
            for fold in range(5):
                path = SCORE_DIR_06 / f"{criterion}__{config}__repeat{repeat}_fold{fold}.csv"
                df = pd.read_csv(path)
                # mean of the 1,000-sample bootstrap distribution -- matches 05's own aggregation convention
                pe = df.groupby("Endpoint")[METRICS].mean()
                row = {"criterion": criterion, "config": config, "repeat": repeat, "fold": fold}
                for m in METRICS:
                    row[f"after_{m}"] = pe.loc["CYP2D6_pIC50_direct_inhibition", m]
                rows.append(row)
after_df = pd.DataFrame(rows)
print(f"after_df: {after_df.shape} (expect (200, 6): 4 configs x 2 criteria x 25 folds)")

summary_05 = pd.read_csv(SUMMARY_05_PATH)

results = []
for criterion in CRITERIA:
    for config in CONFIGS:
        sub_after = (after_df[(after_df["criterion"] == criterion) & (after_df["config"] == config)]
                     .sort_values(["repeat", "fold"]).reset_index(drop=True))
        sub_before = summary_05[summary_05["config"] == config].sort_values(["repeat", "fold"]).reset_index(drop=True)
        assert len(sub_after) == 25 and len(sub_before) == 25, "expected 25 folds per (config, criterion)"
        assert (sub_after[["repeat", "fold"]].to_numpy() == sub_before[["repeat", "fold"]].to_numpy()).all(), \
            "before/after fold order mismatch -- paired test would be invalid"

        for metric in METRICS:
            before = sub_before[f"CYP2D6_{metric}"].to_numpy()
            after = sub_after[f"after_{metric}"].to_numpy()
            delta = after - before

            shapiro_p = float(shapiro(delta).pvalue)
            normal = shapiro_p >= ALPHA
            if normal:
                test_name = "paired t-test"
                _, p_value = ttest_rel(after, before)
            else:
                test_name = "Wilcoxon signed-rank"
                _, p_value = wilcoxon(delta)

            results.append({
                "criterion": criterion, "config": config, "metric": metric,
                "before_mean": before.mean(), "after_mean": after.mean(), "delta_mean": delta.mean(),
                "shapiro_p": shapiro_p, "branch": "parametric" if normal else "nonparametric",
                "test": test_name, "p_value": float(p_value), "significant": bool(p_value < ALPHA),
            })

results_df = pd.DataFrame(results)
results_path = OUT / "before_after_comparison.csv"
results_df.to_csv(results_path, index=False)
print(f"wrote {results_path}")

pd.set_option("display.width", 160)
results_df

after_df: (200, 6) (expect (200, 6): 4 configs x 2 criteria x 25 folds)
wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/06_outlier_check/before_after_comparison.csv


,criterion,config,metric,before_mean,after_mean,delta_mean,shapiro_p,branch,test,p_value,significant
0,residual,chemeleon__rf,ST-RAE,0.937235,0.902073,-0.035161,0.780878,parametric,paired t-test,4.597603e-08,True
1,residual,chemeleon__rf,R2,0.141597,0.135805,-0.005792,0.634752,parametric,paired t-test,1.532456e-01,False
2,residual,chemeleon__lightgbm,ST-RAE,0.948368,0.930195,-0.018174,0.475966,parametric,paired t-test,5.133234e-04,True
3,residual,chemeleon__lightgbm,R2,0.110424,0.106032,-0.004392,0.800465,parametric,paired t-test,4.522349e-01,False
4,residual,ecfp4_narrow__lightgbm,ST-RAE,0.959821,0.946349,-0.013472,0.543563,parametric,paired t-test,5.175929e-02,False
5,residual,ecfp4_narrow__lightgbm,R2,0.099610,0.089870,-0.009740,0.514681,parametric,paired t-test,8.807214e-02,False
6,residual,chemprop_chemeleoninit,ST-RAE,1.003907,0.934307,-0.069600,0.268430,parametric,paired t-test,1.561109e-05,True
7,residual,chemprop_chemeleoninit,R2,0.097991,0.112979,0.014988,0.178114,parametric,paired t-test,2.075844e-01,False
8,ci_width,chemeleon__rf,ST-RAE,0.937235,0.928985,-0.008250,0.334973,parametric,paired t-test,8.834666e-02,False
9,ci_width,chemeleon__rf,R2,0.141597,0.120066,-0.021531,0.423237,parametric,paired t-test,2.086402e-05,True


**What this shows:**

**Criterion 1 (CV-residual, model-independent outliers):** ST-RAE improves after
exclusion in all 4 configs, significantly so in 3 of 4 (`chemeleon__rf` p=4.6e-08,
`chemeleon__lightgbm` p=5.1e-04, `chemprop_chemeleoninit` p=1.6e-05); `ecfp4_narrow__
lightgbm` trends the same direction but falls just short of alpha=0.05 (p=0.052). R2
moves in small, non-significant, mixed directions in every config. **Removing the
compounds the model itself struggled with, pooled across repeats, improves CYP2D6
ST-RAE without a detectable R2 cost, fairly consistently across model families.**

**Criterion 2 (CI-width, measurement-uncertainty outliers):** the picture is
different. ST-RAE reaches significance in only one config -- `chemprop_chemeleoninit`
(p=1.5e-05, a similarly-sized improvement to its own criterion-1 result); the three
tabular configs show small, non-significant, mixed-direction ST-RAE changes. R2,
meanwhile, gets **significantly worse** for all three tabular configs
(`chemeleon__rf` p=2.1e-05, `chemeleon__lightgbm` p=4.2e-06, `ecfp4_narrow__lightgbm`
p=0.017) and is flat for `chemprop_chemeleoninit` (p=0.974). **Removing the
widest-CI compounds does not reliably help CYP2D6 ST-RAE for the tabular configs, and
measurably hurts their R2** -- consistent with the task's own framing that, given
04c's residual/CI-width correlation, this criterion tests "does removing noisy points
help" rather than "why does CYP2D6 underperform," and here the answer for CI-width
specifically is: for three of four configs, no on ST-RAE and actively worse on R2.

`chemprop_chemeleoninit` is the one config that benefits (significantly, on ST-RAE)
from *either* criterion -- suggesting it is sensitive to CYP2D6's hard/noisy
compounds regardless of which of the two criteria identifies them, unlike the tabular
configs, whose R2 the CI-width criterion specifically damages.

Both are genuine, reportable results, not failed runs: criterion 1 is a real,
model-independent-outlier finding that a specific subset of CYP2D6 compounds hurts
ST-RAE across model families; criterion 2 is a real null-to-negative result --
CYP2D6's characteristically wide-CI compounds are not simply prunable noise from the
tabular models' point of view.

## 3. Outcome

- **Criterion 1 (CV-residual) exclusion helps CYP2D6 ST-RAE**: significant
  improvement for `chemeleon__rf`, `chemeleon__lightgbm`, and
  `chemprop_chemeleoninit`; a same-direction, non-significant trend for
  `ecfp4_narrow__lightgbm`. R2 is not detectably affected either way, in any config.
- **Criterion 2 (CI-width) exclusion does not reliably help CYP2D6 ST-RAE**: only
  `chemprop_chemeleoninit` shows a significant ST-RAE improvement; the three tabular
  configs do not. R2 **significantly worsens** for all three tabular configs and is
  flat for `chemprop_chemeleoninit`.
- The two criteria diverge in practical outcome despite a genuine 34.7% overlap in
  which compounds they flag -- most of each criterion's flagged set is unique to it,
  and that non-overlapping majority is where their effects differ.

In [4]:
BASELINE_PRED_DIR = REPO_ROOT / "outputs" / "05_cv_comparison" / "predictions"
OTHER_ISOFORMS = ["CYP1A2_pIC50_direct_inhibition", "CYP2C9_pIC50_direct_inhibition", "CYP3A4_pIC50_direct_inhibition"]

# sanity check only: criterion-1 exclusion masks just the CYP2D6 label for
# chemprop_chemeleoninit's multitask retrains -- confirm the other three isoforms'
# raw predictions didn't meaningfully change, by reading the raw prediction files
# directly (never scored for these isoforms; scoring in this notebook is CYP2D6-only)
pooled = {iso: {"before": [], "after": []} for iso in OTHER_ISOFORMS}
n_total = 0
for repeat in range(5):
    for fold in range(5):
        after_df = pd.read_csv(OUT / "predictions" / f"residual__chemprop_chemeleoninit__repeat{repeat}_fold{fold}.csv")
        before_df = pd.read_csv(BASELINE_PRED_DIR / f"chemprop_chemeleoninit__repeat{repeat}_fold{fold}.csv")
        merged = after_df.merge(before_df, on="inchikey", suffixes=("_after", "_before"))
        assert len(merged) == len(after_df) == len(before_df), \
            f"repeat={repeat} fold={fold}: before/after test-set compounds don't match 1:1"
        n_total += len(merged)
        for iso in OTHER_ISOFORMS:
            pooled[iso]["before"].append(merged[f"{iso}_before"].to_numpy())
            pooled[iso]["after"].append(merged[f"{iso}_after"].to_numpy())

sanity_rows = []
for iso in OTHER_ISOFORMS:
    before = np.concatenate(pooled[iso]["before"])
    after = np.concatenate(pooled[iso]["after"])
    abs_diff = np.abs(after - before)
    sanity_rows.append({
        "isoform": iso.split("_")[0],
        "n_pooled_predictions": len(before),
        "mean_abs_diff": abs_diff.mean(),
        "max_abs_diff": abs_diff.max(),
        "pearson_r_vs_baseline": np.corrcoef(before, after)[0, 1],
    })

sanity_df = pd.DataFrame(sanity_rows)
print(f"pooled predictions across all 25 residual/chemprop_chemeleoninit folds: {n_total}")
sanity_df

pooled predictions across all 25 residual/chemprop_chemeleoninit folds: 24525


,isoform,n_pooled_predictions,mean_abs_diff,max_abs_diff,pearson_r_vs_baseline
0,CYP1A2,24525,0.205799,2.164009,0.921143
1,CYP2C9,24525,0.142548,1.591146,0.953743
2,CYP3A4,24525,0.191946,1.926234,0.953877


## 4. Sanity check: did excluding CYP2D6-only labels leave the other three isoforms alone?

Criterion-1 exclusion for `chemprop_chemeleoninit` masks only the CYP2D6 label for
flagged compounds -- CYP1A2/CYP2C9/CYP3A4 were never scored on these retrains (only
CYP2D6 is in scope for this check), but the raw per-fold prediction files already
contain all four isoforms, so this compares those raw predictions directly against
the 05 baseline predictions for the same compounds -- no scoring pipeline, no
retraining.

**Not near-identical.** Pearson correlation with the 05 baseline is 0.92-0.95 (not
~1.0) on all three untouched isoforms, with mean absolute differences of 0.14-0.21
pIC50 units (max up to ~2.16) pooled across 24,525 predictions. Masking only the
CYP2D6 label did **not** leave CYP1A2/CYP2C9/CYP3A4 predictions unchanged for
`chemprop_chemeleoninit`'s shared-encoder multitask architecture -- flagging this for
investigation, not resolving it here.

## 5. Step 1: does WHICH compounds are masked explain the divergence?

Investigating the previous section's finding (CYP1A2/2C9/3A4 predictions diverge from
the 05 baseline by more than expected, r=0.92-0.95, despite their labels never being
touched by exclusion). Two competing explanations: (a) masking 75 CYP2D6 labels shifts
`chemprop_chemeleoninit`'s shared multitask encoder enough to move the other three
isoforms' predictions too, or (b) `run_5x5_cv_comparison.py`'s own documented deviation
of leaving `OMP_NUM_THREADS` unset (for speed) makes training genuinely
non-deterministic even at a fixed seed, and that noise -- not the exclusion itself --
is what Section 4 measured.

This is a free check on data that already exists: `residual` and `ci_width` mask
substantially different compounds (26 of the 75 flagged overlap between them, 34.7% --
Section 1) while sharing identical seeds and folds otherwise. If (a) is right, two
criteria masking mostly-different compounds should pull the shared encoder in
different, criterion-specific ways, producing a meaningfully different
divergence-from-05 pattern. If (b) is right, both criteria should diverge from 05 by
about the same amount, regardless of which compounds they mask -- because the dominant
source of the difference isn't the masking content at all.

In [5]:
CRITERIA = ["residual", "ci_width"]

# Step 1 (free check, no training): compare residual-after and ci_width-after against
# the 05 baseline separately, on the three isoforms whose labels neither criterion
# touches. These two criteria mask different compounds (only 26 of 75 overlap, per
# Section 1's overlap report) but share seeds/folds otherwise -- if masking WHICH
# compounds drives the divergence found above, the two criteria's divergence-from-05
# should differ meaningfully; if it doesn't, that points toward generic training noise.
step1_records = {}
for criterion in CRITERIA:
    pooled = {iso: {"before": [], "after": []} for iso in OTHER_ISOFORMS}
    for repeat in range(5):
        for fold in range(5):
            after_df = pd.read_csv(OUT / "predictions" / f"{criterion}__chemprop_chemeleoninit__repeat{repeat}_fold{fold}.csv")
            before_df = pd.read_csv(BASELINE_PRED_DIR / f"chemprop_chemeleoninit__repeat{repeat}_fold{fold}.csv")
            merged = after_df.merge(before_df, on="inchikey", suffixes=("_after", "_before"))
            assert len(merged) == len(after_df) == len(before_df), \
                f"{criterion} repeat={repeat} fold={fold}: before/after test-set compounds don't match 1:1"
            for iso in OTHER_ISOFORMS:
                pooled[iso]["before"].append(merged[f"{iso}_before"].to_numpy())
                pooled[iso]["after"].append(merged[f"{iso}_after"].to_numpy())
    for iso in OTHER_ISOFORMS:
        before = np.concatenate(pooled[iso]["before"])
        after = np.concatenate(pooled[iso]["after"])
        abs_diff = np.abs(after - before)
        key = iso.split("_")[0]
        step1_records.setdefault(key, {"isoform": key})
        step1_records[key][f"{criterion}_mean_abs_diff"] = abs_diff.mean()
        step1_records[key][f"{criterion}_pearson_r"] = np.corrcoef(before, after)[0, 1]

step1_df = pd.DataFrame([step1_records[k] for k in ["CYP1A2", "CYP2C9", "CYP3A4"]])
step1_df = step1_df[["isoform", "residual_mean_abs_diff", "residual_pearson_r", "ci_width_mean_abs_diff", "ci_width_pearson_r"]]
step1_df

,isoform,residual_mean_abs_diff,residual_pearson_r,ci_width_mean_abs_diff,ci_width_pearson_r
0,CYP1A2,0.205799,0.921143,0.218679,0.907707
1,CYP2C9,0.142548,0.953743,0.145441,0.948591
2,CYP3A4,0.191946,0.953877,0.212012,0.946651


**This supports (b), generic training noise, not (a) masking-content, as the driver.**
`residual` and `ci_width` mask 74% different compounds (49 of 75 unique to each) yet
produce divergence-from-05 that is nearly the same size and the same per-isoform
pattern on all three untouched isoforms: CYP1A2 is the most divergent under both
criteria (0.206 vs 0.219 mean abs diff, r=0.921 vs 0.908), CYP2C9 the least divergent
under both (0.143 vs 0.145, r=0.954 vs 0.949), CYP3A4 in between under both (0.192 vs
0.212, r=0.954 vs 0.947). If masking-content were driving the shift via a
compound-specific pull on the shared encoder, two criteria masking mostly-different
compounds should not land this close together, nor preserve the same isoform ordering.
This doesn't rule masking-content out as a contributor entirely, but the balance of
this evidence points toward run-to-run training noise as the larger factor.

**Stopping here per instruction.** Step 2 (the repeat-run diagnostic, which trains the
same config twice to test this directly) is not run in this update and needs explicit
go-ahead before any new training -- the numbers above are for the user to review first.

## 6. Step 2: repeat-run determinism check (real training, confirmed by the user)

**Interpretation threshold -- fixed here, before the retrain runs below, for the same
reason Section 1's outlier threshold was fixed before checking CV impact: deciding
after the fact what counts as "close enough" would let the interpretation bend to
whatever comes out.**

- If repeat-vs-repeat divergence for repeat=0/fold=0 lands in the same range Step 1
  found for the real 06-vs-05 comparison -- roughly **0.14-0.22 mean abs diff,
  r=0.91-0.95** -- that **confirms training nondeterminism is a major contributor** to
  what Section 4 measured.
- If it comes back an order of magnitude smaller -- roughly **0.01-0.03 mean abs diff,
  r above ~0.99** -- that **argues nondeterminism alone doesn't explain the full
  divergence**, and something else (most plausibly in the exclusion pipeline itself)
  is also contributing.

**Method:** `chemprop_chemeleoninit`, repeat=0/fold=0, zero exclusion, the exact seed
`generate_5x5_cv_manifest.py` originally assigned that (repeat, fold) -- read directly
from `outputs/05_cv_comparison/manifest.csv`, not retyped. Trained twice, back to
back, identical config and seed both times, via `src.chemprop_screen`'s own
`load_screen_population`/`build_predict_csv`/`build_training_csv`/`run_chemprop_train`/
`run_chemprop_predict` -- the same functions `scripts/cyp2d6_outlier_check.py` and
`scripts/run_5x5_cv_comparison.py` already use, not a new implementation. Each run is
a fresh `chemprop train`/`chemprop predict` subprocess, exactly matching how the real
25-fold runs are launched -- no in-process state carries over between the two runs.


In [6]:
import logging

from src.chemprop_screen import (
    build_predict_csv, build_training_csv, load_screen_population,
    run_chemprop_predict, run_chemprop_train, verify_predictions,
)
from src.vendor.openadmet_eval.config import REGRESSION_ENDPOINTS

FOLDS_PATH = REPO_ROOT / "data" / "folds" / "cv_folds.csv"
CURATED_PATH = REPO_ROOT / "data" / "processed" / "train_inhibition_curated.csv"
VAL_FRACTION = 0.15  # matches run_5x5_cv_comparison.py's own VAL_FRACTION
CHEMPROP_ARCHITECTURE_ARGS = ["--from-foundation", "CHEMELEON", "--multi-hot-atom-featurizer-mode", "V2"]
CHEMPROP_EPOCHS = 50
CHEMPROP_PATIENCE = 5
DETERMINISM_DIR = OUT / "chemprop_runs" / "determinism_check"

manifest = pd.read_csv(REPO_ROOT / "outputs" / "05_cv_comparison" / "manifest.csv")
seed_row = manifest[(manifest["config"] == "chemprop_chemeleoninit") & (manifest["repeat"] == 0) & (manifest["fold"] == 0)]
assert len(seed_row) == 1, "expected exactly one manifest row for chemprop_chemeleoninit repeat=0 fold=0"
seed = int(seed_row["seed"].iloc[0])
print(f"repeat=0 fold=0 seed (read from outputs/05_cv_comparison/manifest.csv, same as 05's original run): {seed}")

# File-only logger (no stream handler): chemprop's own subprocess output is piped
# through this logger line-by-line (src/chemprop_screen.py's run_subprocess_streamed),
# which would otherwise dump thousands of raw progress-bar/epoch lines into this
# notebook's saved output. Full detail still goes to disk, just not embedded here.
logger = logging.getLogger("step2_determinism_check")
logger.setLevel(logging.INFO)
logger.handlers.clear()
log_path = REPO_ROOT / "logs" / "06_step2_determinism_check.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
file_handler = logging.FileHandler(log_path, mode="a")
file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(file_handler)


def run_baseline_once(run_tag, seed, logger):
    """Zero-exclusion chemprop_chemeleoninit fit for repeat=0/fold=0, via the same
    src.chemprop_screen functions run_5x5_cv_comparison.py and cyp2d6_outlier_check.py
    already use -- no new training implementation. Resumable the same way as those
    scripts: skips retraining if this run_tag's raw_predictions.csv already exists."""
    run_dir = DETERMINISM_DIR / run_tag
    raw_pred_csv = run_dir / "raw_predictions.csv"
    if raw_pred_csv.exists():
        print(f"{run_tag}: raw_predictions.csv already exists -- loading, not retraining")
        return pd.read_csv(raw_pred_csv)

    population = load_screen_population(FOLDS_PATH, CURATED_PATH, "repeat_0", 0, VAL_FRACTION, seed, logger)
    run_dir.mkdir(parents=True, exist_ok=True)
    predict_csv = build_predict_csv(population, run_dir, logger)
    train_csv = run_dir / "train_input.csv"
    build_training_csv(population, REGRESSION_ENDPOINTS, train_csv, logger, require_all_targets=False)
    run_chemprop_train(
        train_csv, REGRESSION_ENDPOINTS, run_dir, logger,
        CHEMPROP_ARCHITECTURE_ARGS, CHEMPROP_EPOCHS, CHEMPROP_PATIENCE, seed,
    )
    run_chemprop_predict(run_dir / "model_0", predict_csv, raw_pred_csv, logger)
    expected_names = set(population.loc[population["screen_split"] == "screen_test", "Molecule_Name"])
    verify_predictions(raw_pred_csv, expected_names, REGRESSION_ENDPOINTS, logger)
    return pd.read_csv(raw_pred_csv)


run1_df = run_baseline_once("run1", seed, logger)
run2_df = run_baseline_once("run2", seed, logger)

merged = run1_df.merge(run2_df, on="inchikey", suffixes=("_run1", "_run2"))
assert len(merged) == len(run1_df) == len(run2_df), "run1/run2 test-set compounds don't match 1:1"

determinism_rows = []
for endpoint in REGRESSION_ENDPOINTS:
    a = merged[f"{endpoint}_run1"].to_numpy()
    b = merged[f"{endpoint}_run2"].to_numpy()
    abs_diff = np.abs(a - b)
    determinism_rows.append({
        "isoform": endpoint.split("_")[0],
        "n": len(a),
        "mean_abs_diff": abs_diff.mean(),
        "max_abs_diff": abs_diff.max(),
        "pearson_r": np.corrcoef(a, b)[0, 1],
    })

determinism_df = pd.DataFrame(determinism_rows)
determinism_df

repeat=0 fold=0 seed (read from outputs/05_cv_comparison/manifest.csv, same as 05's original run): 2684470948
run1: raw_predictions.csv already exists -- loading, not retraining
run2: raw_predictions.csv already exists -- loading, not retraining


,isoform,n,mean_abs_diff,max_abs_diff,pearson_r
0,CYP1A2,981,0.0,0.0,1.0
1,CYP2C9,981,0.0,0.0,1.0
2,CYP2D6,981,0.0,0.0,1.0
3,CYP3A4,981,0.0,0.0,1.0


**Result: falls on the "doesn't fully explain it" side of the pre-stated threshold --
in fact further than the example range given.** Two independent, back-to-back
`chemprop_chemeleoninit` fits on repeat=0/fold=0, identical config and seed, zero
exclusion, produced **mean_abs_diff = 0.0 and pearson_r = 1.0 exactly on all four
isoforms** -- not just small, but bit-identical (confirmed independently: both runs'
`model_0/best.pt` checkpoints share the same MD5 hash, and `raw_predictions.csv` is a
byte-for-byte match between the two runs). This is smaller than the 0.01-0.03 /
r-above-~0.99 range that would already have counted as "doesn't fully explain it" --
on this machine, for this config and seed, chemprop training is fully deterministic
across two sequential, unloaded runs.

**This does not support run-to-run training nondeterminism as the explanation for
Section 4's divergence.** Per the pre-stated interpretation: something in the
exclusion pipeline itself -- beyond generic training noise -- is the more likely
contributor to the CYP1A2/2C9/3A4 shift found in Section 4 and confirmed
criterion-independent in Section 5. This is an **open item, not resolved here**: the
training-CSV construction for the masked-label runs (does masking one task's target
for 75 compounds change anything else about the resulting CSV -- row order, dtype,
`chemprop_split` assignment -- beyond the intended NaN) is the next place to look
before treating Section 3's CYP2D6 significance as a clean causal result.

## 7. Direct diff: what did masking change in `train_input.csv`?

Section 6 ruled out training nondeterminism (Step 2: two identical back-to-back
`chemprop_chemeleoninit` fits produced byte-identical checkpoints). So something about
how the excluded run's training data was actually built is doing more than the
intended CYP2D6-label mask. Direct diagnostic only -- no retraining, no new chemprop
runs -- comparing the two runs' real `train_input.csv` files already on disk from the
actual runs (not reconstructed), for one representative case:
criterion=`residual`, config=`chemprop_chemeleoninit`, repeat=0, fold=0.

- (a) baseline: `outputs/05_cv_comparison/chemprop_runs/chemprop_chemeleoninit__repeat0_fold0/train_input.csv`
- (b) excluded: `outputs/06_outlier_check/chemprop_runs/residual__chemprop_chemeleoninit__repeat0_fold0/train_input.csv`

`train_input.csv` itself only carries `canonical_smiles` (no `Molecule_Name`/
`inchikey`), so recovering compound identity for the checks below means joining back
to `train_inhibition_curated.csv` on `canonical_smiles` -- verified unique across all
4,905 curated compounds first, not assumed, before relying on it as a join key.

Four independent checks, reported one at a time as the task specified -- no averaging
across them, a failure in any one is independently actionable.

In [7]:
BASELINE_PATH = REPO_ROOT / "outputs" / "05_cv_comparison" / "chemprop_runs" / "chemprop_chemeleoninit__repeat0_fold0" / "train_input.csv"
EXCLUDED_PATH = OUT / "chemprop_runs" / "residual__chemprop_chemeleoninit__repeat0_fold0" / "train_input.csv"
CURATED_PATH = REPO_ROOT / "data" / "processed" / "train_inhibition_curated.csv"
FLAGGED_PATH = OUT / "flagged_compounds" / "criterion1_residual_flagged.csv"

baseline = pd.read_csv(BASELINE_PATH)
excluded = pd.read_csv(EXCLUDED_PATH)
curated = pd.read_csv(CURATED_PATH)
flagged = pd.read_csv(FLAGGED_PATH)

# train_input.csv itself carries no Molecule_Name/inchikey -- canonical_smiles is the
# only identifier in the file, verified unique across all 4,905 curated compounds
# first (not assumed) before using it as the join key for every check below.
assert curated["canonical_smiles"].is_unique, "canonical_smiles is not a safe join key"
flagged_smiles = set(curated.loc[curated["inchikey"].isin(flagged["inchikey"]), "canonical_smiles"])
assert len(flagged_smiles) == len(flagged), "not every flagged compound resolved to a canonical_smiles"

merged = baseline.merge(excluded, on="canonical_smiles", suffixes=("_before", "_after"), how="outer", indicator=True)
print(f"(a) baseline: {len(baseline)} rows")
print(f"(b) excluded: {len(excluded)} rows")
print(f"outer-merged on canonical_smiles: {len(merged)} rows")

(a) baseline: 3924 rows
(b) excluded: 3924 rows
outer-merged on canonical_smiles: 3924 rows


In [8]:
# CHECK 1 -- row order
smiles_to_name = curated.set_index("canonical_smiles")["Molecule_Name"]
baseline_order = baseline["canonical_smiles"].map(smiles_to_name).reset_index(drop=True)
excluded_order = excluded["canonical_smiles"].map(smiles_to_name).reset_index(drop=True)

row_order_identical = (baseline_order == excluded_order).all()
print(f"row order identical, position-by-position by Molecule_Name: {row_order_identical}")
if not row_order_identical:
    mismatches = (baseline_order != excluded_order)
    print(f"first mismatched positions: {mismatches[mismatches].index[:10].tolist()}")

row order identical, position-by-position by Molecule_Name: True


**CHECK 1 -- ROW ORDER: PASS.** Every row's `Molecule_Name` (recovered via the
`canonical_smiles` join) matches position-by-position between (a) and (b), across all
3,924 rows. Masking the CYP2D6 label did not reorder the file.

In [9]:
# CHECK 2 -- split assignment for every compound NOT among the 75 flagged
non_flagged = merged[~merged["canonical_smiles"].isin(flagged_smiles)]
split_mismatch = non_flagged[non_flagged["chemprop_split_before"] != non_flagged["chemprop_split_after"]]

print(f"non-flagged compounds compared: {len(non_flagged)}")
print(f"chemprop_split mismatches: {len(split_mismatch)}")
if len(split_mismatch):
    print(split_mismatch[["canonical_smiles", "chemprop_split_before", "chemprop_split_after"]])

non-flagged compounds compared: 3865
chemprop_split mismatches: 0


**CHECK 2 -- SPLIT ASSIGNMENT: PASS.** All 3,865 non-flagged compounds have an
identical `chemprop_split` value (`screen_inner_train` / `screen_inner_val`) in (a)
and (b) -- zero mismatches. Masking does not interact with train/val split
assignment for the compounds it doesn't touch.

In [10]:
# CHECK 3 -- NaN / dtype handling for the 75 flagged compounds
from src.features import assign_screen_split

manifest = pd.read_csv(REPO_ROOT / "outputs" / "05_cv_comparison" / "manifest.csv")
seed = int(manifest.loc[(manifest["config"] == "chemprop_chemeleoninit") & (manifest["repeat"] == 0) & (manifest["fold"] == 0), "seed"].iloc[0])
cv_folds = pd.read_csv(REPO_ROOT / "data" / "folds" / "cv_folds.csv")
split_df = assign_screen_split(cv_folds, repeat_col="repeat_0", test_fold=0, val_fraction=0.15, seed=seed)
flagged_split_counts = split_df[split_df["inchikey"].isin(flagged["inchikey"])]["screen_split"].value_counts()
print(f"75 flagged compounds' screen_split breakdown for repeat=0/fold=0: {flagged_split_counts.to_dict()}")

flagged_rows = merged[merged["canonical_smiles"].isin(flagged_smiles)]
print(f"-> {len(flagged_rows)} of 75 land in this fold's train_input.csv (inner_train + inner_val); the rest are this fold's own screen_test compounds, never candidates for this fold's training pool regardless of masking\n")

cyp2d6_before = flagged_rows["CYP2D6_pIC50_direct_inhibition_before"]
cyp2d6_after = flagged_rows["CYP2D6_pIC50_direct_inhibition_after"]
print(f"of those {len(flagged_rows)}: CYP2D6 had a real label before masking, all of them: {cyp2d6_before.notna().all()}")
print(f"of those {len(flagged_rows)}: CYP2D6 is NaN after masking, all of them: {cyp2d6_after.isna().all()}")

other_cols = ["CYP1A2_pIC50_direct_inhibition", "CYP2C9_pIC50_direct_inhibition", "CYP3A4_pIC50_direct_inhibition", "chemprop_split"]
for col in other_cols:
    b, a = flagged_rows[f"{col}_before"], flagged_rows[f"{col}_after"]
    identical = ((b == a) | (b.isna() & a.isna())).all()
    print(f"{col} byte-identical to baseline for all flagged rows: {identical}")

print(f"\ndtypes identical across the two full files: {(baseline.dtypes == excluded.dtypes).all()}")
print(baseline.dtypes.to_dict())

75 flagged compounds' screen_split breakdown for repeat=0/fold=0: {'screen_inner_train': 41, 'screen_inner_val': 18, 'screen_test': 16}
-> 59 of 75 land in this fold's train_input.csv (inner_train + inner_val); the rest are this fold's own screen_test compounds, never candidates for this fold's training pool regardless of masking

of those 59: CYP2D6 had a real label before masking, all of them: True
of those 59: CYP2D6 is NaN after masking, all of them: True
CYP1A2_pIC50_direct_inhibition byte-identical to baseline for all flagged rows: True
CYP2C9_pIC50_direct_inhibition byte-identical to baseline for all flagged rows: True
CYP3A4_pIC50_direct_inhibition byte-identical to baseline for all flagged rows: True
chemprop_split byte-identical to baseline for all flagged rows: True

dtypes identical across the two full files: True
{'canonical_smiles': dtype('O'), 'CYP1A2_pIC50_direct_inhibition': dtype('float64'), 'CYP2C9_pIC50_direct_inhibition': dtype('float64'), 'CYP2D6_pIC50_direct_inhi

**CHECK 3 -- NaN / DTYPE HANDLING: PASS**, with one clarifying detail. Only 59 of
the 75 globally-flagged compounds land in *this specific fold's* `train_input.csv`
(41 `screen_inner_train` + 18 `screen_inner_val`); the other 16 are this fold's own
`screen_test` compounds for repeat=0 -- never candidates for this fold's training
pool regardless of masking, confirmed directly via `assign_screen_split` rather than
assumed from the 75/59 gap. Of the 59 that are present: CYP2D6 had a real label
before masking for all of them, and is NaN after masking for all of them, as
intended. Every other column (CYP1A2/2C9/3A4 labels, `chemprop_split`) is
byte-identical to the baseline row for the same compound -- no collateral changes.
Dtypes are identical across the two full files (`float64` for every pIC50 column,
`object` for `canonical_smiles`/`chemprop_split`) -- introducing NaN into the CYP2D6
column did not silently upcast or otherwise change any other column's dtype.

In [11]:
# CHECK 4 -- row count / exact compound set
print(f"(a) baseline rows: {len(baseline)}, (b) excluded rows: {len(excluded)}")
print(f"merge indicator counts: {merged['_merge'].value_counts().to_dict()}")
print(f"compounds only in baseline (dropped in excluded): {(merged['_merge'] == 'left_only').sum()}")
print(f"compounds only in excluded (added, not in baseline): {(merged['_merge'] == 'right_only').sum()}")

(a) baseline rows: 3924, (b) excluded rows: 3924
merge indicator counts: {'both': 3924, 'left_only': 0, 'right_only': 0}
compounds only in baseline (dropped in excluded): 0
compounds only in excluded (added, not in baseline): 0


**CHECK 4 -- ROW COUNT: PASS.** Both files have exactly 3,924 rows. An outer merge
on `canonical_smiles` produces zero `left_only` and zero `right_only` rows -- the
excluded run's `train_input.csv` contains exactly the same 3,924 compounds as the
baseline's, nothing added or dropped beyond the intended masking.

### All four checks pass cleanly

Row order, split assignment, NaN handling, and row count are all identical between
the baseline and residual-excluded `train_input.csv` for repeat=0/fold=0, except for
the single intended change (CYP2D6 masked to NaN for the 59 flagged compounds present
in this fold's training pool). **The cause is NOT in `train_input.csv`
construction.** It must be somewhere else -- `predict_input.csv` construction, or
chemprop's own internal handling of a multitask target with more NaN entries than
before. Not speculating further in this pass, per the task scope: this narrows the
search, it doesn't resolve it.

## 8. Closing statement: what this investigation established, and what it didn't

**Elimination chain** (each step already confirmed above, not re-argued here):

1. **Seed/config nondeterminism** -- ruled out (Section 6): two identical
   back-to-back `chemprop_chemeleoninit` fits, same seed, zero exclusion, produced
   byte-identical checkpoints and predictions.
2. **Training-CSV construction** -- ruled out (Section 7): row order, split
   assignment, NaN/dtype handling, and row count are all identical to baseline for
   repeat=0/fold=0 except the intended CYP2D6 masking itself.
3. **Masking-content specificity as the sole driver** -- ruled out (Section 5):
   `residual` and `ci_width` mask 74% different compounds yet produce near-identical
   divergence magnitude and isoform ordering from the 05 baseline (Section 4's
   original finding).

**Leading hypothesis, not directly confirmed:** chemprop's multitask loss most
likely normalizes or weights each task's contribution by how many valid (non-NaN)
labels that task has in a batch. Masking 59 (in this fold) of CYP2D6's labels
doesn't just remove training signal for CYP2D6 -- it also changes CYP2D6's effective
weight in the combined multitask loss, which could shift the shared encoder's
gradient in a way that reaches CYP1A2/2C9/3A4 even though their own labels were
never altered. This is consistent with everything ruled out above, but it has not
been directly verified against chemprop's actual loss implementation. It is the
leading hypothesis, not a confirmed mechanism.

**Implication for Section 2's headline CYP2D6 result:** the measured CYP2D6
improvement after exclusion is real and directly observed -- that is not in
question. What is in question is the causal story behind it: exclusion's effect on
CYP2D6 may be partly a genuine data-quality effect (removing compounds the model
could not predict well) and partly an implicit reweighting effect (changing CYP2D6's
share of the shared multitask loss, independent of which compounds were removed).
This notebook cannot currently separate those two contributions from each other.
That is a caveat on how to interpret the result, not a retraction of it -- Section
2's significance testing stands as reported.

**This investigation is closed.** No further diagnostic work is planned in this
notebook. If the reweighting mechanism needs to be directly confirmed later, that is
a separate, deliberately scoped follow-up -- e.g. reading chemprop's loss-weighting
implementation directly, or checking whether divergence magnitude scales with the
number of masked labels per fold -- not an open thread left hanging off this one.